# Evidently AI Lab 2: Monitoring a Bike-Sharing Demand Model

In Lab 1 we compared two halves of one dataset that we split on purpose, so drift was guaranteed.
Here we monitor a **deployed regression model over time**, which is closer to what monitoring means in production.

**Scenario.** A model predicts hourly bike rentals (`count`) from weather and calendar features.
We train it on the first four months of 2011 and then "deploy" it. Each following month arrives as a new production batch.
Evidently answers two questions for every batch:

1. **Data drift:** do the input features still look like the reference data?
2. **Model performance:** is the model still accurate (MAE, MAPE, R²)?

**Contents**
1. Load data and define time batches
2. Train the model and build the reference
3. Inspect one batch in depth (drift and regression reports)
4. Test suites: pass/fail rules
5. Monitor every month (Jul 2011 to Dec 2012)
6. Fault injection: can Evidently catch a broken sensor?
7. Respond: retrain and compare

> **Environment:** see `README.md`. In short: Python 3.11 or 3.12, then `pip install -r requirements.txt`.
> The first run downloads the dataset from OpenML, so it needs internet access.

In [ ]:
# Environment check: fail early with a clear message instead of a confusing import error later.
import sys
from importlib import metadata

py = sys.version_info
if not ((3, 11) <= (py.major, py.minor) <= (3, 12)):
    raise RuntimeError(f"Python {py.major}.{py.minor} is not supported. Use Python 3.11 or 3.12 (see README.md).")
if int(metadata.version("pandas").split(".")[0]) >= 3:
    raise RuntimeError("pandas 3.x is not supported by evidently 0.7.0. Run: pip install -r requirements.txt")
for pkg in ["evidently", "pandas", "numpy", "scikit-learn", "matplotlib"]:
    print(f"{pkg:<14}{metadata.version(pkg)}")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.ensemble import RandomForestRegressor

from evidently import Dataset, DataDefinition, Regression, Report
from evidently.presets import DataDriftPreset, RegressionPreset
from evidently.metrics import MAE, MAPE, R2Score, DriftedColumnsCount
from evidently.tests import lte, Reference

REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(exist_ok=True)

## 1. Load data and define time batches
The Bike Sharing Demand dataset (UCI, via OpenML) has hourly rentals for 2011 and 2012.
Rows are in chronological order and `year` is `0` (2011) or `1` (2012).
We build a `period` index (0 = Jan 2011 to 23 = Dec 2012) and use calendar months as batches.

In [ ]:
bikes = datasets.fetch_openml("Bike_Sharing_Demand", version=2, as_frame=True).frame
for c in ["season", "holiday", "workingday", "weather"]:
    bikes[c] = bikes[c].astype(str)

bikes["period"] = bikes["year"] * 12 + bikes["month"] - 1
bikes["month_label"] = (
    (2011 + bikes["year"]).astype(str) + "-" + bikes["month"].astype(str).str.zfill(2)
)
print(bikes.shape)
bikes.head()

In [ ]:
# Yearly demand level: this is a real trend the model will not know about.
bikes.groupby("year")["count"].agg(["mean", "median", "max"])

## 2. Train the model and build the reference
| Role | Months | Used for |
|---|---|---|
| Train | Jan to Apr 2011 | fit the model |
| **Reference** | May to Jun 2011 | held-out data the model has *not* seen; defines "normal" |
| Production | Jul 2011 to Dec 2012 | monthly batches we monitor |

The reference is held out on purpose. It shows how the model performs on unseen but in-distribution data.

In [ ]:
NUM = ["hour", "weekday", "temp", "feel_temp", "humidity", "windspeed"]
CAT = ["weather", "workingday", "holiday"]
TARGET, PRED = "count", "prediction"


def features(frame):
    return pd.get_dummies(frame[NUM + CAT]).astype(float)


X_all = features(bikes)
train_mask = bikes["period"] <= 3

model = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1)
model.fit(X_all[train_mask], bikes.loc[train_mask, TARGET])
bikes[PRED] = model.predict(X_all)

reference = bikes[bikes["period"].isin([4, 5])].reset_index(drop=True)
print("train rows:", int(train_mask.sum()), "| reference rows:", len(reference))

In [ ]:
# Tell Evidently the column roles. The `regression` mapping marks target and prediction.
schema = DataDefinition(
    numerical_columns=NUM + [PRED],
    categorical_columns=CAT,
    regression=[Regression(target=TARGET, prediction=PRED)],
)


def to_dataset(frame):
    cols = NUM + CAT + [TARGET, PRED]
    return Dataset.from_pandas(frame[cols].reset_index(drop=True), data_definition=schema)


ref_ds = to_dataset(reference)

## 3. Inspect one batch in depth
We compare **July 2011**, the first production month, against the reference with two presets:
`DataDriftPreset` (per-column drift tests) and `RegressionPreset` (error metrics and plots).

In [ ]:
jul_2011 = bikes[bikes["month_label"] == "2011-07"]

report = Report([DataDriftPreset(), RegressionPreset()])
snap = report.run(to_dataset(jul_2011), ref_ds)
snap.save_html(str(REPORT_DIR / "batch_2011-07.html"))
snap  # renders the interactive report in Jupyter

In [ ]:
def flat_metrics(snapshot):
    """Turn a snapshot into a tidy table of metric name, statistic and value."""
    rows = []
    for m in snapshot.dict()["metrics"]:
        name = m["metric_id"].split("(")[0]
        arg = m["metric_id"].split("column=")[1].rstrip(")") if "column=" in m["metric_id"] else ""
        v = m["value"]
        if isinstance(v, dict):
            for k, x in v.items():
                rows.append((f"{name} {arg}".strip(), k, x))
        else:
            rows.append((f"{name} {arg}".strip(), "value", v))
    return pd.DataFrame(rows, columns=["metric", "stat", "value"])


flat_metrics(snap)

In [ ]:
# Each column gets its own test. The description names the method and threshold used.
tests = pd.DataFrame([
    {"column": t["name"].replace("Value Drift for column ", ""),
     "drifted": getattr(t["status"], "value", str(t["status"])) == "FAIL",
     "detail": t["description"]}
    for t in snap.dict()["tests"] if t["name"].startswith("Value Drift for column ")
])
tests

## 4. Test suites: pass/fail rules
A report describes a batch. **Tests** turn it into a decision.
Here we encode two monitoring rules:

* **Data rule:** at most 30% of columns may drift.
* **Model rule:** MAE may be at most 50% worse than the reference MAE.

`Reference(relative=0.5)` means "reference value plus 50%".

In [ ]:
def run_tests(current_frame, reference_ds=ref_ds):
    suite = Report([
        DriftedColumnsCount(share_tests=[lte(0.3)]),
        MAE(mean_tests=[lte(Reference(relative=0.5))]),
    ], include_tests=False)
    snap = suite.run(to_dataset(current_frame), reference_ds)
    out = []
    for t in snap.dict()["tests"]:
        out.append({"rule": t["name"], "status": getattr(t["status"], "value", str(t["status"])),
                    "detail": t["description"]})
    return pd.DataFrame(out)


pd.set_option("display.max_colwidth", 120)
for label in ["2011-07", "2012-12"]:
    print(label)
    display(run_tests(bikes[bikes["month_label"] == label]))

## 5. Monitor every month
We loop over all 18 production batches and record, per month:

* the **drift share** and a drifted / not drifted flag for each column,
* **MAE, MAPE and R²** of the model,
* an **alert** when a rule is broken. A *data alert* is drift share above 30%. A *performance alert* is MAE above 1.5x the reference MAE.


In [ ]:
DRIFT_SHARE_LIMIT = 0.3
MAE_FACTOR = 1.5


def evaluate_batch(frame, reference_ds, ref_mae):
    # include_tests=True gives one pass/fail test per column; FAIL means that column drifted.
    # The raw drift score is NOT comparable across columns (p-value for some, distance for others).
    snap = Report([DataDriftPreset(), MAE(), MAPE(), R2Score()], include_tests=True).run(
        to_dataset(frame), reference_ds)
    d = snap.dict()
    drift_flags = {}
    for t in d["tests"]:
        if t["name"].startswith("Value Drift for column "):
            col = t["name"].replace("Value Drift for column ", "")
            drift_flags[col] = getattr(t["status"], "value", str(t["status"])) == "FAIL"
    row = {}
    for m in d["metrics"]:
        mid, v = m["metric_id"], m["value"]
        if mid.startswith("DriftedColumnsCount"):
            row["drifted_cols"], row["drift_share"] = int(v["count"]), v["share"]
        elif mid.startswith("MAE"):
            row["mae"] = v["mean"]
        elif mid.startswith("MAPE"):
            row["mape_pct"] = v["mean"]
        elif mid.startswith("R2Score"):
            row["r2"] = v
    row["data_alert"] = row["drift_share"] > DRIFT_SHARE_LIMIT
    row["perf_alert"] = row["mae"] > MAE_FACTOR * ref_mae
    row["drifted_names"] = ", ".join(c for c, f in drift_flags.items() if f)
    return row, drift_flags


ref_mae = float(np.mean(np.abs(reference[TARGET] - reference[PRED])))
print(f"Reference MAE: {ref_mae:.1f} bikes/hour  ->  performance alert above {MAE_FACTOR * ref_mae:.1f}")

In [ ]:
prod_months = sorted(bikes.loc[bikes["period"] >= 6, "month_label"].unique())
rows, scores = [], {}
for label in prod_months:
    batch = bikes[bikes["month_label"] == label]
    row, sc = evaluate_batch(batch, ref_ds, ref_mae)
    row.update(month=label, rows=len(batch), mean_actual=batch[TARGET].mean(), mean_pred=batch[PRED].mean())
    rows.append(row)
    scores[label] = sc

monitor = pd.DataFrame(rows).set_index("month")
monitor[["rows", "drifted_cols", "drift_share", "mae", "mape_pct", "r2", "mean_actual", "mean_pred",
         "data_alert", "perf_alert", "drifted_names"]].round(2)

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
x = range(len(monitor))

ax[0].bar(x, monitor["drift_share"], color=np.where(monitor["data_alert"], "#d1495b", "#4c78a8"))
ax[0].axhline(DRIFT_SHARE_LIMIT, ls="--", c="k", lw=1)
ax[0].set_ylabel("share of drifted columns")
ax[0].set_title("Data drift (red = data alert)")

ax[1].plot(x, monitor["mae"], marker="o", c="#4c78a8")
ax[1].axhline(ref_mae, c="g", lw=1, label="reference MAE")
ax[1].axhline(MAE_FACTOR * ref_mae, ls="--", c="k", lw=1, label=f"alert at {MAE_FACTOR}x")
ax[1].set_ylabel("MAE (bikes/hour)")
ax[1].set_title("Model error")
ax[1].legend()

ax[2].plot(x, monitor["mean_actual"], marker="o", label="actual mean")
ax[2].plot(x, monitor["mean_pred"], marker="o", label="predicted mean")
ax[2].set_ylabel("mean count")
ax[2].set_title("Actual vs predicted demand")
ax[2].legend()

ax[2].set_xticks(list(x))
ax[2].set_xticklabels(monitor.index, rotation=60)
plt.tight_layout()
plt.savefig(REPORT_DIR / "monitoring_timeline.png", dpi=120)
plt.show()

In [ ]:
# Which columns drift, and when? Dark cell = Evidently flagged that column as drifted that month.
heat = pd.DataFrame(scores).T.astype(float)
fig, ax = plt.subplots(figsize=(11, 5))
ax.imshow(heat.T.values, aspect="auto", cmap="Reds", vmin=0, vmax=1)
ax.set_yticks(range(len(heat.columns)))
ax.set_yticklabels(heat.columns)
ax.set_xticks(range(len(heat.index)))
ax.set_xticklabels(heat.index, rotation=60)
ax.set_title("Drifted columns by month (dark = drift detected)")
plt.tight_layout()
plt.show()

### What to read from this
* **The data alert fires in 17 of 18 months.** `temp`, `feel_temp` and `humidity` are flagged in every single month. Our reference covers only May and June, so any other season looks "drifted". With this reference the dataset-level data alert is mostly seasonal noise, not a useful signal.
* **The performance alert tells a different story.** In 2011 the inputs had already drifted for months, yet MAE stayed at about 47 to 50 (the reference is 45.8). The error only crosses the 1.5x line in Feb 2012 and reaches about 145 in Sep and Oct 2012.
* **The cause is a shift in the target, not in the inputs.** The model under-predicts mean demand in every 2012 month, by about 50 to 140 bikes per hour (the bottom chart). Average demand grew from about 144 per hour in 2011 to about 235 in 2012, and a model trained on early 2011 cannot know that. This is *concept drift*.
* **Input drift did not predict the failure.** Catching it needed the true `count`, which means ground-truth labels. Monitor both inputs and outputs, and expect labels to arrive with some delay.
* **Try this:** use a full year as the reference instead of May to Jun and see how many data alerts remain.

> **Reading drift scores:** Evidently picks the test by sample size. With a few hundred rows it uses p-values, where *small* means drift. With larger samples it uses distances, where *large* means drift. Compare per-column drift flags, not raw scores. The per-column test table in section 3 shows the method and threshold used for each column.

## 6. Fault injection: can Evidently catch a broken sensor?
Real seasonal drift is hard to judge by eye. To check that the monitor works, we need a batch that is *known* to be clean and then break it deliberately.

* **Control batch:** a random half of the reference months, which is held out from training and drawn from the same distribution as the reference data.
* **Faults:** miscalibrated temperature sensor, a stuck humidity sensor, wind speed logged in the wrong unit.

In [ ]:
rng = np.random.RandomState(42)
mask = rng.rand(len(reference)) < 0.5
ref_a, control = reference[mask].reset_index(drop=True), reference[~mask].reset_index(drop=True)
ref_a_ds = to_dataset(ref_a)
ref_a_mae = float(np.mean(np.abs(ref_a[TARGET] - ref_a[PRED])))


def predict_with(frame):
    """Re-run the model on (possibly corrupted) features, as production would."""
    out = frame.copy()
    out[PRED] = model.predict(features(out).reindex(columns=X_all.columns, fill_value=0.0))
    return out


faults = {
    "control (no fault)": control,
    "temp sensor +8 C": control.assign(temp=control["temp"] + 8, feel_temp=control["feel_temp"] + 8),
    "humidity stuck at 0.5": control.assign(humidity=0.5),
    "windspeed x3.6 (m/s to km/h)": control.assign(windspeed=control["windspeed"] * 3.6),
}

fault_rows = []
for name, frame in faults.items():
    frame = predict_with(frame)
    row, _ = evaluate_batch(frame, ref_a_ds, ref_a_mae)
    fault_rows.append({"scenario": name, "drifted_cols": row["drifted_cols"],
                       "drift_share": round(row["drift_share"], 2), "mae": round(row["mae"], 1),
                       "data_alert": row["data_alert"], "perf_alert": row["perf_alert"],
                       "drifted columns": row["drifted_names"] or "-"})
pd.DataFrame(fault_rows).set_index("scenario")

What the table shows:

* **No false positives.** The control batch has zero drifted columns.
* **Per-column drift finds the fault exactly.** Each fault flags its own column: `temp` and `feel_temp` for the temperature shift, `humidity` for the stuck sensor, `windspeed` for the unit error. `prediction` also drifts when the corrupted input changes the model output.
* **Dataset-level rules miss them.** The drifted share is 0.1 to 0.3, never above the 0.3 limit, so `data_alert` stays off. MAE does not worsen either (for the temperature fault it even improves slightly), so `perf_alert` is off too.

A single broken sensor therefore hides below a dataset-wide threshold and does not hurt accuracy, but it is still a data-quality problem. In practice, add column-level alerts for the features you trust most.

## 7. Respond: retrain and compare
When monitoring says the model is stale, the usual response is to retrain on more recent data.
We retrain on all of 2011 and compare both models on every 2012 month.

In [ ]:
train_2011 = bikes["year"] == 0
model_v2 = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1)
model_v2.fit(X_all[train_2011], bikes.loc[train_2011, TARGET])
bikes["prediction_v2"] = model_v2.predict(X_all)

y12 = bikes[bikes["year"] == 1]
cmp = (y12.assign(err_v1=(y12[TARGET] - y12[PRED]).abs(), err_v2=(y12[TARGET] - y12["prediction_v2"]).abs())
          .groupby("month_label")[["err_v1", "err_v2"]].mean()
          .rename(columns={"err_v1": "MAE v1 (trained Jan-Apr 2011)", "err_v2": "MAE v2 (trained all 2011)"}))
cmp.round(1)

In [ ]:
ax = cmp.plot(marker="o", figsize=(9, 4), title="2012 MAE: stale model vs retrained model")
ax.axhline(ref_mae, c="g", lw=1, label="reference MAE")
ax.legend()
plt.ylabel("MAE (bikes/hour)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Retraining on all of 2011 lowers the error by roughly 17 to 25% in every 2012 month, but it does **not** bring the model back to the reference level of about 46.
By Sep and Oct 2012 the retrained model is still about 2.5x worse than the reference.
Demand kept growing after the training window closed, so a one-off retrain only narrows the gap.
Realistic fixes are periodic retraining on recent data, or adding a trend feature. Monitoring is what tells you when to do it.

## Summary
| Idea | Where in this lab |
|---|---|
| Reference vs production batches over time | sections 2 and 5 |
| Data drift and model performance as separate signals | sections 3 and 5 |
| Rules with pass/fail tests | section 4 |
| Validating the monitor with known faults | section 6 |
| Closing the loop with retraining | section 7 |

Reports are saved to `reports/` (HTML and PNG). They are not committed because the HTML files are about 4 MB each.